# PyTorch / Lightning Shakespeare Transformer — Build Guide

## Goal
Build a PyTorch + Lightning transformer using standard ecosystem tools,
keeping BPC on `tiny_val` comparable to the hand-rolled version.

---

## Tokenization — HuggingFace `tokenizers`

Train a BPE tokenizer from scratch on `full_train.txt` with a configurable vocab size.
`vocab_size = 128 + n_merges` gives the same effective control as the hand-rolled `train_bpe`.
The tokenizer is trained once and saved to disk; load it for all subsequent runs.
BPC remains comparable across tokenizers since it normalizes back to characters —
compute `avg_chars_per_token` once on the training set and store it in Config.

---

## Data Loading — PyTorch `DataLoader`

Tokenize the split `.txt` files with the trained BPE tokenizer, chunk into sequences
of length `n`, and wrap in a PyTorch `DataLoader`. The `DataLoader` handles shuffling,
batching, and parallel prefetch. This replaces `load_data` from the hand-rolled version.

---

## Model — `nn.Module` + Lightning

The architecture is unchanged (see Architecture Overview below). Wrap it in a
`LightningModule` which implements `training_step`, `validation_step`, and
`configure_optimizers`. The `Trainer` handles the epoch loop, gradient clipping,
and device placement.

---

## Logging — Weights & Biases

Replace `log_results` / `training_log.csv` with `wandb`. The Lightning `WandbLogger`
integrates automatically inside `training_step` / `validation_step` via `self.log()`.
Still call `log_results` from `utils.py` at the end to keep the shared
`training_log.csv` up to date for apples-to-apples BPC comparison.

---

## Checkpointing — `ModelCheckpoint` + `EarlyStopping`

Pass both as callbacks to `Trainer`. `ModelCheckpoint` saves the best-val checkpoint
automatically; `EarlyStopping` stops training when val loss stops improving. Use
`save_model` from `utils.py` at the end of training to write a checkpoint in the
shared `vX-Y_name` format so it appears in the same namespace as hand-rolled runs.

---

## Apples-to-Apples BPC Checklist
- [ ] Compute `avg_chars_per_token` from the HF tokenizer on `tiny_val.txt`
- [ ] Evaluate on `tiny_val.txt` after restoring best-val checkpoint
- [ ] `bpc = cross_entropy_loss / log(2) / avg_chars_per_token`
- [ ] Write result to `training_log.csv` via `log_results` with a `name`

## Imports

In [19]:
import torch
import wandb
import os
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from torch.utils.data import DataLoader
from utils import log_results, save_model
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import WandbLogger

In [20]:
from dataclasses import dataclass

@dataclass
class Config:
    """Hyperparameters and derived values for the Transformer. d_k and d_v are
    computed automatically from d_model and H."""
    B: int = 32         # batch size
    n: int = 256        # sequence length (context window)
    d_model: int = 384  # embedding / hidden dimension
    d_ff: int = 1536    # feed-forward inner dimension (typically 4 × d_model)
    H: int = 8          # number of attention heads
    epochs: int = 100
    layers: int = 6
    lr: float = 3e-4
    drop_prob: float = 0.1
    vocab_size: int = 1000  # 128 = char-level; >128 = BPE

    def __post_init__(self):
        # d_k = d_model / H; each head attends over a d_k-dimensional subspace
        self.d_k = self.d_model // self.H
        self.d_v = self.d_k


## Tokenizer & Data

In [21]:
def train_tokenizer(train_path, vocab_size=1000, save_path="tokenizer.json"):
    """Train a BPE tokenizer on the training corpus and save it to disk.

    vocab_size = 128 base ASCII chars + n_merges, so vocab_size=628 ~ 500 merges.
    Only needs to be run once; reload with Tokenizer.from_file(save_path) after.
    Returns tokenizer
    """
    tokenizer = Tokenizer(BPE())
    trainer = BpeTrainer(vocab_size=vocab_size)
    tokenizer.train(files=[train_path], trainer=trainer)
    tokenizer.save(save_path)
    return tokenizer
   


def load_tokenizer(path="tokenizer.json"):
    return Tokenizer.from_file(path)

In [22]:


def make_dataloaders(path, tokenizer, cfg, shuffle=False):

    """Tokenize text files, chunk into sequences of length n, return DataLoaders.

    Steps:
        - Read text, encode with tokenizer
        - Split into non-overlapping chunks of length n
        - convert to tensors
        - Return (train_loader, val_loader) with shuffle=True for train
    """
    

    with open(path) as f:
        text = f.read()

    ids = tokenizer.encode(text).ids
    
    tokens = torch.tensor(ids)
    n_chunks = len(tokens) // cfg.n
    truncated_tokens = tokens[:n_chunks * cfg.n]
    tokens_matrix = truncated_tokens.view(n_chunks, cfg.n)
    data_loader = DataLoader(tokens_matrix, batch_size = cfg.B, shuffle=shuffle, drop_last=True, num_workers=4)
    return data_loader



## Model

## Architecture Overview

**Ownership tree** — `Transformer` owns everything; the others are internal sub-modules:

```
Transformer
├── nn.Embedding           (token → d_model)
├── TransformerBlock × N
│   ├── nn.LayerNorm       (pre-attention norm)
│   ├── CausalSelfAttention
│   │   └── nn.Linear × 4  (Wq, Wk, Wv, Wo)
│   ├── nn.LayerNorm       (pre-FFN norm)
│   └── nn.Linear × 2      (FFN in/out)
├── nn.LayerNorm           (final norm)
└── nn.Linear              (lm_head, weight-tied to Embedding)
```

**Data flow through `Transformer.forward`:**

1. `(B, n)` token IDs → `nn.Embedding` → `(B, n, d_model)`
2. Loop through N `TransformerBlock`s — each does two residual additions:
   - `x = x + attn(norm1(x))` — attention sub-layer
   - `x = x + ffn(norm2(x))` — FFN sub-layer
3. Final `nn.LayerNorm` → `lm_head` → `(B, n, vocab_size)` logits
4. Shift by one: `logits[:, :-1]` predicts `targets[:, 1:]`
5. Flatten and pass to `F.cross_entropy`

**Inside `CausalSelfAttention.forward`:**

Project Q/K/V → reshape to `(B, H, n, d_k)` → apply RoPE to Q and K →
`F.scaled_dot_product_attention` → reshape back to `(B, n, d_model)` → output projection Wo

Because everything is registered as `nn.Module` sub-modules, `model.to(device)`,
`model.parameters()`, and `model.state_dict()` automatically reach all of them.


In [23]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with RoPE positional encoding.

    Uses F.scaled_dot_product_attention (Flash Attention on CUDA) for the inner
    QKV computation. RoPE is applied to Q and K before the attention call.
    Dropout is passed directly to scaled_dot_product_attention rather than
    applied as a separate layer.
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.Wq = nn.Linear(cfg.d_model, cfg.d_model)
        self.Wk = nn.Linear(cfg.d_model, cfg.d_model)
        self.Wv = nn.Linear(cfg.d_model, cfg.d_model)
        self.Wo = nn.Linear(cfg.d_model, cfg.d_model)
        
        thetas = 1 / 10000 ** (torch.arange(0, cfg.d_k, 2) / cfg.d_k)
        M = torch.arange(cfg.n)
        angles = torch.outer(M, thetas)  # (n, d_k/2)
        self.register_buffer('sin', angles.sin())
        self.register_buffer('cos', angles.cos())

    def _apply_rope(self, X):
        """Apply Rotary Position Embedding to X ~ (B, H, n, d_k).

        Splits d_k into even/odd index pairs and rotates each pair by a
        position-dependent angle: [x1, x2] → [x1·cos - x2·sin, x1·sin + x2·cos].
        This encodes relative position into the dot-product attention score.
        """
        x1 = X[..., 0::2]  # even indices
        x2 = X[..., 1::2]  # odd indices
        x1r = x1 * self.cos - x2 * self.sin
        x2r = x1 * self.sin + x2 * self.cos
        # Interleave rotated pairs back into the original dimension order
        Xr = torch.stack([x1r, x2r], dim=-1)
        Xr = Xr.flatten(-2)
        return Xr


    def forward(self, X):
        cfg = self.cfg
        B, n = X.shape[0], X.shape[1]
        Q = self.Wq(X)
        K = self.Wk(X)
        V = self.Wv(X)

        Q = Q.view(B, n, cfg.H, cfg.d_k).transpose(-3, -2)
        K = K.view(B, n, cfg.H, cfg.d_k).transpose(-3, -2)
        V = V.view(B, n, cfg.H, cfg.d_v).transpose(-3, -2)

        Q = self._apply_rope(Q)
        K = self._apply_rope(K)

        O = F.scaled_dot_product_attention(Q, K, V, is_causal=True, dropout_p=cfg.drop_prob if self.training else 0.0)
        O = O.transpose(-3, -2).reshape(B, n, cfg.H * cfg.d_v)
        X_out = self.Wo(O)
        return X_out

In [24]:
class TransformerBlock(nn.Module):
    """One decoder block: pre-norm attention sub-layer + pre-norm FFN sub-layer.

    Pre-norm means LayerNorm is applied to the residual stream before each sub-layer:
    x = x + sublayer(norm(x)), not norm(x + sublayer(x)).
    """
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff),
            nn.GELU(),
            nn.Linear(cfg.d_ff, cfg.d_model),
            nn.Dropout(cfg.drop_prob),
        )
        

    def forward(self, X):
        Xnorm = self.norm1(X)
        Xattn = self.attn(Xnorm)
        X = X + Xattn
        Xnorm2 = self.norm2(X)
        Xff = self.ff(Xnorm2)
        X = X + Xff
        return X

    

In [25]:
class Transformer(nn.Module):
    """Decoder-only transformer for character/BPE-level language modelling.

    Architecture mirrors transformer.ipynb exactly:
      embedding -> N x TransformerBlock -> LayerNorm -> lm_head (weight-tied to embedding)

    forward() takes a (B, n) token tensor and returns (logits, targets) where targets
    is the input shifted by one position — ready to pass straight to F.cross_entropy.
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.output_norm = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight  # weight tying, transpose is done implicitly by PyTorch
        

    def forward(self, Xin):
        """Returns (logits, targets). logits shape: (B*(n-1), vocab_size)."""
        X = self.embedding(Xin)
        for block in self.blocks:
            X = block(X)
        X = self.output_norm(X)
        X = self.lm_head(X)
        logits = X[:, :-1, :]         # drop prediction at last position (no target for it)
        targets = Xin[:, 1:]
        return logits, targets   



    def generate(self, prompt_ids, max_new_tokens=200, temperature=1.0, top_k=None):
        """Autoregressively sample max_new_tokens tokens given a list of prompt token IDs."""
        X = torch.tensor(prompt_ids).unsqueeze(0).to(next(self.parameters()).device) #unsqueeze adds a batch dimension of size 1, works on cuda or cpu

        
        
        for _ in range(max_new_tokens):
            X_crop = X[:, -self.cfg.n:] if X.shape[1] > self.cfg.n else X
            h = self.embedding(X_crop)
            for block in self.blocks:
                h = block(h)
            h = self.output_norm(h)
            h = self.lm_head(h)
            logits = h[0, -1, :]
            logits = logits / temperature
            if top_k is not None:
                # Mask all tokens below the k-th highest logit before softmax
                threshold = torch.topk(logits, top_k).values[-1]
                logits[logits < threshold] = float('-inf')
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            X = torch.cat([X, next_token.unsqueeze(0)], dim=1)
        return X[0, len(prompt_ids):].tolist()



## Training

In [26]:
class ShakespeareModule(L.LightningModule):
    """LightningModule wrapping Transformer for use with Trainer.

    training_step and validation_step log loss to wandb via self.log().
    configure_optimizers returns AdamW + ReduceLROnPlateau, matching transformer.ipynb.
    """
    def __init__(self, cfg):
        super().__init__()
        self.model = Transformer(cfg)
        self.cfg = cfg
        self.loss_fn = torch.nn.CrossEntropyLoss()
        

    def training_step(self, batch, _):
        """Forward pass on one batch; return loss. self.log() sends to wandb."""
        logits, targets = self.model(batch) #nn.Module says when you call a module, it calls the "forward" function
        loss = self.loss_fn(logits.reshape(-1, self.cfg.vocab_size), targets.reshape(-1))
        self.log("train_loss", loss, on_epoch=True, on_step=False) #defined in L.LightningModule
        return loss
        

    def validation_step(self, batch, _):
        """Same as training_step but no gradient (handled by Lightning). Logged as val_loss."""
        logits, targets = self.model(batch)
        loss = self.loss_fn(logits.reshape(-1, self.cfg.vocab_size), targets.reshape(-1))
        self.log("val_loss", loss, on_epoch=True)
        

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.cfg.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"},
        }

In [27]:
class EpochPrinter(L.Callback):  ##AI Slop
    def __init__(self):
        self._train_losses = []

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        loss = outputs["loss"] if isinstance(outputs, dict) else outputs
        self._train_losses.append(loss.item())
        n_batches = trainer.num_training_batches
        pl_module.print(f"Epoch {trainer.current_epoch} | "
                        f"Loss {loss.item():.4f} | "
                        f"Batch {batch_idx + 1} / {n_batches}",
                        end="\r", flush=True)

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        train_loss = sum(self._train_losses) / len(self._train_losses) if self._train_losses else float("nan")
        self._train_losses = []
        val_loss = trainer.callback_metrics.get("val_loss", float("nan"))
        line = f"Epoch {trainer.current_epoch:>3}  train: {train_loss:.4f}  val: {val_loss:.4f}"
        pl_module.print(f"\r{line.ljust(60)}", flush=True)

class StopOnFile(L.Callback):
    STOP_FILE = "/teamspace/studios/this_studio/ml_models/shakespeare_generator/stop_training"

    def on_train_epoch_end(self, trainer, pl_module):
        if os.path.exists(self.STOP_FILE):
            trainer.should_stop = True
            os.remove(self.STOP_FILE)

def run(cfg, train_path, val_path, note=""):
    """Full training run: tokenize → build dataloaders → train.

    Steps:
      1. Load or train tokenizer
      2. Build train/val DataLoaders via make_dataloaders
      3. Init ShakespeareModule
      4. Build Trainer with WandbLogger, ModelCheckpoint, EarlyStopping callbacks
      5. trainer.fit(module, train_loader, val_loader)

    Returns (trainer, module, tokenizer) for post-training eval.
    """

    tokenizer_path = "tokenizer.json"

    try:
        tokenizer = load_tokenizer(path=tokenizer_path)
        assert len(tokenizer.get_vocab()) == cfg.vocab_size
    except:
        tokenizer = train_tokenizer(train_path=train_path, vocab_size=cfg.vocab_size, save_path=tokenizer_path)

    train_loader = make_dataloaders(path=train_path, tokenizer=tokenizer, cfg=cfg, shuffle=True)
    val_loader = make_dataloaders(path=val_path, tokenizer=tokenizer, cfg=cfg, shuffle=False)

    module = ShakespeareModule(cfg)

    trainer = L.Trainer(
        max_epochs=cfg.epochs,
        callbacks=[
            EpochPrinter(),
            StopOnFile(),
            ModelCheckpoint(monitor="val_loss", save_top_k=1, filename="best"),
            EarlyStopping(monitor="val_loss", patience=4),
        ],
        logger=WandbLogger(project="shakespeare"),
        accelerator="gpu",
        devices=1,
        gradient_clip_val=1.0,
        enable_progress_bar=False,
    )
    trainer.fit(module, train_loader, val_loader)

    return trainer, module, tokenizer

In [28]:
cfg = Config()
wandb.finish()  # close any leftover run from a previous interrupted session
trainer, module, tokenizer = run(cfg, "../data/full_train_minus_tiny_val.txt", "../data/full_val.txt", note="lightning train")
wandb.finish()
# run this command in the terminal to end the train

# touch ~/ml_models/shakespeare_generator/stop_training


epoch,▁▁▂▂▂▂▃▃▃▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
train_loss,█▂▂▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▂▂▂▂▃▃▃▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
val_loss,█▄▃▃▂▂▂▁▁▂▁▁▁
epoch,12
train_loss,4.89562
trainer/global_step,2157
val_loss,4.80431


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ Transformer      │ 11.0 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 11.0 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.0 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 84                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Epoch   0  train: 18.5629  val: 6.0107                      
Epoch   1  train: 5.3501  val: 4.8615                       
Epoch   2  train: 4.7129  val: 4.5435                       
Epoch   3  train: 4.4332  val: 4.3846                       
Epoch   4  train: 4.2317  val: 4.2523                       
Epoch   5  train: 4.0661  val: 4.1379                       
Epoch   6  train: 3.9330  val: 4.0456                       
Epoch   7  train: 3.8208  val: 3.9973                       
Epoch   8  train: 3.7292  val: 3.9657                       
Epoch   9  train: 3.6460  val: 3.9250                       
Epoch  10  train: 3.5742  val: 3.9114                       
Epoch  11  train: 3.5088  val: 3.8904                       
Epoch  12  train: 3.4502  val: 3.8707                       
Epoch  13  train: 3.3929  val: 3.8781                       
Epoch  14  train: 3.3402  val: 3.8873                       
Epoch  15  train: 3.2899  val: 3.8653                       
Epoch  16  train: 3.2401

In [30]:
# Step 6: evaluate tiny_val with best checkpoint
import glob, os

tiny_val_path = "../data/tiny_val.txt"

try:
    best_path = trainer.checkpoint_callback.best_model_path
    wandb_name = trainer.logger.experiment.name
except NameError:
    ckpts = glob.glob("lightning_logs/**/best.ckpt", recursive=True)
    best_path = max(ckpts, key=os.path.getmtime)
    wandb_name = ""
    print(f"Loading checkpoint: {best_path}")

module = ShakespeareModule.load_from_checkpoint(best_path, cfg=cfg)
module.eval().to("cuda")

tiny_val_loader = make_dataloaders(tiny_val_path, tokenizer, cfg, shuffle=False)
losses = []
with torch.no_grad():
    for batch in tiny_val_loader:
        logits, targets = module.model(batch.to("cuda"))
        loss = module.loss_fn(logits.reshape(-1, cfg.vocab_size), targets.reshape(-1))
        losses.append(loss.item())
tiny_val_loss = sum(losses) / len(losses)

# Step 7: compute BPC and log to shared training_log.csv
with open(tiny_val_path) as f:
    tiny_val_text = f.read()
tiny_val_ids = tokenizer.encode(tiny_val_text).ids
avg_chars_per_token = len(tiny_val_text) / len(tiny_val_ids)

try:
    val_loss = trainer.checkpoint_callback.best_model_score.item()
except NameError:
    val_loss = float("nan")

log_results(cfg, train_loss=0.0, val_loss=val_loss, tiny_val_loss=tiny_val_loss,
            avg_chars_per_token=avg_chars_per_token, name=wandb_name,
            runs_path="../training_runs.txt", csv_path="../training_log.csv")
save_model(module.model, cfg, directory="../checkpoints", name=wandb_name)


Run: 2026-03-13 20:15:14  [balmy-spaceship-9]
layers=6, d_model=384, d_ff=1536, H=8
B=32, n=256, lr=0.0003, vocab_size=1000
avg_chars_per_token=2.454

Final  | train:     0.0000  bpc: 0.0000
       | full_val:  3.8611  bpc: 2.2698
       | tiny_val:  3.5075  bpc: 2.0619
Time   | total: 0.0s  avg/epoch: 0.0s

Logged to ../training_runs.txt
CSV row appended to ../training_log.csv
Saved model to ../checkpoints/balmy-spaceship-9.pt


('../checkpoints/balmy-spaceship-9.pt', 'balmy-spaceship-9')